# 01 · Data merging & cleaning
Builds the canonical NFHS-5 district table and the pooled HMIS maternal-health table, reconciles district names, and engineers features (blueprint §2). Re-extraction of the HMIS archive takes several minutes, so it is skipped when the interim CSV already exists.

In [ ]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.configuration import load_config, configured_path, feature_columns, zones
config = load_config()
P = lambda name: configured_path(config, name)
pd.set_option("display.max_columns", 40)

In [ ]:
from src.data.nfhs_extract import build_nfhs_canonical
from src.data.hmis_extract import pool_hmis_annual_files
REBUILD = False
if REBUILD or not P("nfhs_raw").exists():
    build_nfhs_canonical(P("nfhs_source"), P("nfhs_raw"))
hmis_files = [P(f"hmis_{y}") for y in config["real_data"]["hmis_years"]]
if REBUILD or not P("hmis_raw").exists():
    pool_hmis_annual_files(hmis_files, P("hmis_raw"), progress=True)
nfhs, hmis = pd.read_csv(P("nfhs_raw")), pd.read_csv(P("hmis_raw"))
print("NFHS districts:", len(nfhs), "| HMIS districts:", len(hmis), "| HMIS archives present:", [f.name for f in hmis_files if f.exists()])

## District reconciliation
Within-state matching: aliases → exact normalised name → `rapidfuzz` token-sort (≥ 90 auto, 75–89 manual review). HMIS districts carved out after NFHS-5 are summed into their NFHS parent.

In [ ]:
from src.data.merge_nfhs_hmis import merge_sources
m = config["matching"]
master = merge_sources(P("nfhs_raw"), P("hmis_raw"), ROOT / m["aliases_path"], P("match_audit"),
                       m["automatic_match_threshold"], m["review_match_threshold"], config["real_data"]["minimum_matched_share"])
audit = pd.read_csv(P("match_audit"))
print(f"Matched share: {master['hmis_matched'].mean():.1%}")
audit["method"].value_counts()

In [ ]:
audit.loc[audit["method"].isin(["unmatched", "manual_review", "hmis_unused", "fuzzy_auto"])]

## Feature engineering & missingness

In [ ]:
from src.data.preprocess_impute import engineer_features
master = engineer_features(master, feature_columns(), config["preprocessing"])
missing = master[feature_columns()].isna().mean().sort_values(ascending=False)
missing.head(10).to_frame("missing share")

In [ ]:
master[feature_columns()].describe().T[["count", "mean", "50%", "min", "max"]].round(2)